Imports and File path

In [10]:
import time 
import pandas as pd

pd.set_option('display.float_format', "{:.2f}".format)
DATA_PATH = "data/raw/paysim.csv"

Load the data

In [11]:
start = time.time()
df = pd.read_csv(DATA_PATH, engine="pyarrow", dtype_backend="pyarrow")
print(f"Loaded in {time.time() - start:.1f} seconds")

Loaded in 12.6 seconds


size,types and memory

In [12]:
print("Rows:", f"{len(df):,}", "| Columns:", df.shape[1])
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
df.dtypes

Rows: 6,362,620 | Columns: 11
Memory: 664 MB


step               int64[pyarrow]
type              string[pyarrow]
amount            double[pyarrow]
nameOrig          string[pyarrow]
oldbalanceOrg     double[pyarrow]
newbalanceOrig    double[pyarrow]
nameDest          string[pyarrow]
oldbalanceDest    double[pyarrow]
newbalanceDest    double[pyarrow]
isFraud            int64[pyarrow]
isFlaggedFraud     int64[pyarrow]
dtype: object

In [13]:
df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.00,0.00,0,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.00,0.00,0,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.00,0.00,0,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.00,0.00,0,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.00,40348.79,0,0


Missing values and duplicates

In [14]:
print("Missing values per column:")
print(df.isna().sum())
print("\nFully duplicated rows:", f"{df.duplicated().sum():,}")

Missing values per column:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Fully duplicated rows: 0


In [15]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
step,6362620.00,<NA>,<NA>,<NA>,243.40,142.33,1.00,156.00,239.00,335.00,743.00
type,6362620,5,CASH_OUT,2237500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,6362620.00,<NA>,<NA>,<NA>,179861.90,603858.23,0.00,13389.57,74871.94,208721.48,92445516.64
nameOrig,6362620,6353307,C2098525306,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceOrg,6362620.00,<NA>,<NA>,<NA>,833883.10,2888242.67,0.00,0.00,14208.00,107315.18,59585040.37
newbalanceOrig,6362620.00,<NA>,<NA>,<NA>,855113.67,2924048.50,0.00,0.00,0.00,144258.41,49585040.37
nameDest,6362620,2722362,C1286084959,113,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceDest,6362620.00,<NA>,<NA>,<NA>,1100701.67,3399180.11,0.00,0.00,132705.66,943036.71,356015889.35
newbalanceDest,6362620.00,<NA>,<NA>,<NA>,1224996.40,3674128.94,0.00,0.00,214661.44,1111909.25,356179278.92
isFraud,6362620.00,<NA>,<NA>,<NA>,0.00,0.04,0.00,0.00,0.00,0.00,1.00


In [16]:
print(df["type"].value_counts(), "\n")
print("Origin prefixes:\n", df["nameOrig"].str[0].value_counts(), "\n")
print("Destination prefixes:\n", df["nameDest"].str[0].value_counts())

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64[pyarrow] 

Origin prefixes:
 nameOrig
C    6362620
Name: count, dtype: int64[pyarrow] 

Destination prefixes:
 nameDest
C    4211125
M    2151495
Name: count, dtype: int64[pyarrow]


In [17]:
fraud = df["isFraud"].sum()
print(f"Fraud rows: {fraud:,} ({fraud / len(df):.4%})")
print(f"Flagged by legacy rule (isFlaggedFraud): {df['isFlaggedFraud'].sum():,}")
print(f"Steps: {df['step'].min()} to {df['step'].max()} ({df['step'].nunique()} distinct)")

Fraud rows: 8,213 (0.1291%)
Flagged by legacy rule (isFlaggedFraud): 16
Steps: 1 to 743 (743 distinct)


## Session 1 findings: data health
- 6,362,620 rows, 11 columns; no missing values; no duplicate rows.
- All origins are customers (C). Destinations: 4,211,125 customers, 2,151,495 merchants (M).
- Merchant destinations equal the PAYMENT count exactly: all payments go to merchants, and only payments do.
- All 743 hourly steps present (1 to 743), so no gaps in the timeline.
- 8,213 fraud rows (0.129%): heavy class imbalance.
- Legacy rule (isFlaggedFraud) flags only 16 rows, catching at most ~0.2% of fraud.